In [ ]:
import os
import time
import matplotlib.pyplot as plt
import numpy as np
import pathlib as pl
import shutil
import sys

import flopy
from modflowapi import ModflowApi
from modflowapi.extensions import ApiSimulation

from bmi.wrapper import BMIWrapper

import pyswmm
from pyswmm import Simulation, Nodes
from pyswmm import Output

In [ ]:
sys.path.append("../../common")
from liss_settings import \
    libmf6, \
    get_dflow_control_path, \
    get_dflow_grid_name, get_dflow_dtuser, \
    get_modflow_coupling_tag, get_modflow_grid_name, \
    silent, verbosity, \
    print_path, print_value

In [ ]:
control_path = get_dflow_control_path(domain="GP", resolution="medium")

In [ ]:
grid_name = get_dflow_grid_name(control_path)
print(grid_name)

In [ ]:
dflowfm_dtuser = get_dflow_dtuser(control_path)
print(dflowfm_dtuser)

#### Setup and initialize D-FLOW FM

You will need to set `dflow_dirpath` to the correct directory on your machine.

In [ ]:
# dflow_dirpath = os.path.abspath(r"C:\Program Files\Deltares\Delft3D FM Suite 2023.02 HM\plugins\DeltaShell.Dimr\kernels\x64\dflowfm\bin")
# dflow_deps_dirpath = (
#     os.path.abspath(r"C:\Program Files\Deltares\Delft3D FM Suite 2023.02 HM\plugins\DeltaShell.Dimr\kernels\x64\share\bin"),
# )
dflow_basepath = control_path.parent.parent.parent.resolve()
dflow_dirpath = dflow_basepath / "dflowfm_dll.2026.01" 
dflow_base = control_path.parent
dflow_working = control_path.parent
dflow_config = control_path

# # dflow_dirpath = dflow_basepath / "dflowfm_dll.2025.01" 
# dflow_base = dflow_basepath / r"coarse\tides_atm\base"
# dflow_working = dflow_basepath / r"coarse\tides_atm\run"
# dflow_config = dflow_working / "FlowFM.mdu"

In [ ]:
# if dflow_working.is_dir():
#     shutil.rmtree(dflow_working)
# shutil.copytree(dflow_base, dflow_working)
(dflow_working / "output").mkdir(parents=True, exist_ok=True)

In [ ]:
# Add dflowfm dll folder to PATH so that it can be found by the BMIWrapper
os.environ["PATH"] = (
    str(dflow_dirpath) + os.pathsep + os.environ["PATH"]
)

In [ ]:
print_path()

In [ ]:
(pl.Path(dflow_dirpath) / "dflowfm.dll").is_file()

#### Initialize D-Flow FM

In [ ]:
dflowfm = BMIWrapper(
    engine="dflowfm",
    configfile=str(dflow_config),
)

In [ ]:
dflowfm.initialize()

#### Run each time step

In [ ]:
current_time = dflowfm.get_current_time()
end_time = dflowfm.get_end_time()
start_time = current_time

In [ ]:
print(
    f"DFLOWFM current_time: {current_time:15,.1f} sec. ({current_time/86400.:15,.1f} days)\n"
     + f"DFLOWFM end_time:     {end_time:15,.1f} sec. ({end_time/86400.:15,.1f} days)\n"
     + f"DFLOWFM sim time:     {end_time - current_time:15,.1f} sec. ({(end_time - current_time)/86400.:15,.1f} days)"
)

In [ ]:
idx = 0
jdx = 0
t0 = time.perf_counter()

while current_time <= end_time:
    idx += 1
    ontime = dflowfm.get_current_time()
    onday = (ontime - start_time) / 86400.
    dflowfm.update()

    current_time = dflowfm.get_current_time()
    frac_comp = (current_time - start_time) / (end_time - start_time)
    print(f"Current time: {current_time:15,.1f} ({onday:10.3f} days) - {frac_comp:6.2%} complete - ({idx:03d})    ", end="\r")
    
    if current_time == end_time:
        break

vextcum = dflowfm.get_var("vextcum")

t1 = time.perf_counter()
print(f"\nrun time: {(t1 - t0) / 60.} min")

#### Finalize models

In [ ]:
dflowfm.finalize()